In [10]:
# Recreate player_stats.py with combined batting/bowling table
from pathlib import Path

player_stats_code = '''"""
Module for extracting and storing player statistics from T20 matches.
"""

import sqlite3
from pathlib import Path
from typing import List, Dict, Any, Optional
from .models import MatchData, Innings
from collections import defaultdict


def extract_combined_player_stats_from_match(match: MatchData) -> List[Dict[str, Any]]:
    """Extract combined batting and bowling statistics for all players in a match's first innings."""
    if not match.innings or len(match.innings) < 1:
        return []
    
    first_innings = match.innings[0]
    match_id = str(getattr(match.meta, 'data_version', 'unknown'))
    match_date = match.info.dates[0] if hasattr(match.info, 'dates') and match.info.dates else 'unknown'
    venue = match.info.venue or 'Unknown'
    
    batting_team = first_innings.team
    bowling_team = None
    if hasattr(match.info, 'teams') and match.info.teams:
        teams = match.info.teams
        bowling_team = teams[1] if teams[0] == batting_team else teams[0]
    
    # Extract batting stats
    batting_stats = {}
    for over in first_innings.overs:
        for delivery in over.deliveries:
            batter = delivery.batter
            if batter not in batting_stats:
                batting_stats[batter] = {'runs': 0, 'balls_faced': 0, 'fours': 0, 'sixes': 0, 'dots_faced': 0, 'dismissed': False}
            
            runs_by_batter = delivery.runs.batter
            batting_stats[batter]['runs'] += runs_by_batter
            
            if not delivery.extras or 'wides' not in delivery.extras:
                batting_stats[batter]['balls_faced'] += 1
            
            if runs_by_batter == 4:
                batting_stats[batter]['fours'] += 1
            elif runs_by_batter == 6:
                batting_stats[batter]['sixes'] += 1
            
            if runs_by_batter == 0 and (not delivery.extras or not delivery.extras.get('wides')):
                batting_stats[batter]['dots_faced'] += 1
            
            if delivery.wickets:
                for wicket in delivery.wickets:
                    if wicket.player_out == batter:
                        batting_stats[batter]['dismissed'] = True
    
    # Extract bowling stats
    bowling_stats = {}
    for over in first_innings.overs:
        for delivery in over.deliveries:
            bowler = delivery.bowler
            if bowler not in bowling_stats:
                bowling_stats[bowler] = {'balls_bowled': 0, 'runs_conceded': 0, 'wickets': 0, 'wides': 0, 'noballs': 0, 'dots_bowled': 0}
            
            if not delivery.extras or ('wides' not in delivery.extras and 'noballs' not in delivery.extras):
                bowling_stats[bowler]['balls_bowled'] += 1
            
            bowling_stats[bowler]['runs_conceded'] += delivery.runs.total
            
            if delivery.wickets:
                for wicket in delivery.wickets:
                    if wicket.kind and wicket.kind.lower() != 'run out':
                        bowling_stats[bowler]['wickets'] += 1
            
            if delivery.extras:
                if 'wides' in delivery.extras:
                    bowling_stats[bowler]['wides'] += delivery.extras['wides']
                if 'noballs' in delivery.extras:
                    bowling_stats[bowler]['noballs'] += delivery.extras['noballs']
            
            if delivery.runs.total == 0:
                bowling_stats[bowler]['dots_bowled'] += 1
    
    # Combine stats for all players
    all_players = set(batting_stats.keys()) | set(bowling_stats.keys())
    result = []
    
    for player in all_players:
        player_data = {'match_id': match_id, 'date': match_date, 'venue': venue, 'player': player}
        
        if player in batting_stats:
            bstats = batting_stats[player]
            player_data.update({
                'batting_team': batting_team,
                'runs': bstats['runs'],
                'balls_faced': bstats['balls_faced'],
                'fours': bstats['fours'],
                'sixes': bstats['sixes'],
                'dots_faced': bstats['dots_faced'],
                'dismissed': 1 if bstats['dismissed'] else 0,
                'strike_rate': round((bstats['runs'] / bstats['balls_faced'] * 100), 2) if bstats['balls_faced'] > 0 else None
            })
        else:
            player_data.update({'batting_team': None, 'runs': None, 'balls_faced': None, 'fours': None, 'sixes': None, 'dots_faced': None, 'dismissed': None, 'strike_rate': None})
        
        if player in bowling_stats:
            bowstats = bowling_stats[player]
            overs_bowled = bowstats['balls_bowled'] // 6
            balls_in_partial_over = bowstats['balls_bowled'] % 6
            overs = overs_bowled + (balls_in_partial_over / 10)
            
            player_data.update({
                'bowling_team': bowling_team,
                'overs': round(overs, 1) if overs > 0 else None,
                'balls_bowled': bowstats['balls_bowled'],
                'runs_conceded': bowstats['runs_conceded'],
                'wickets': bowstats['wickets'],
                'wides': bowstats['wides'],
                'noballs': bowstats['noballs'],
                'dots_bowled': bowstats['dots_bowled'],
                'economy': round((bowstats['runs_conceded'] / overs), 2) if overs > 0 else None
            })
        else:
            player_data.update({'bowling_team': None, 'overs': None, 'balls_bowled': None, 'runs_conceded': None, 'wickets': None, 'wides': None, 'noballs': None, 'dots_bowled': None, 'economy': None})
        
        result.append(player_data)
    
    return result


def extract_first_innings_combined_stats(matches: List[MatchData]) -> List[Dict[str, Any]]:
    """Extract combined batting and bowling statistics from first innings of all matches."""
    all_stats = []
    for match in matches:
        match_stats = extract_combined_player_stats_from_match(match)
        all_stats.extend(match_stats)
    return all_stats


def save_player_stats_to_db(stats: List[Dict[str, Any]], db_path: Path, table_name: str = 'first_innings_player_stats', verbose: bool = True) -> None:
    """Save combined player statistics (batting and bowling) to SQLite database."""
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            match_id TEXT, date TEXT, venue TEXT, player TEXT,
            batting_team TEXT, runs INTEGER, balls_faced INTEGER, fours INTEGER, sixes INTEGER, dots_faced INTEGER, dismissed INTEGER, strike_rate REAL,
            bowling_team TEXT, overs REAL, balls_bowled INTEGER, runs_conceded INTEGER, wickets INTEGER, wides INTEGER, noballs INTEGER, dots_bowled INTEGER, economy REAL
        )
    """)
    
    cursor.execute(f"DELETE FROM {table_name}")
    
    cursor.executemany(f"""
        INSERT INTO {table_name} (match_id, date, venue, player, batting_team, runs, balls_faced, fours, sixes, dots_faced, dismissed, strike_rate, bowling_team, overs, balls_bowled, runs_conceded, wickets, wides, noballs, dots_bowled, economy)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [(s['match_id'], s['date'], s['venue'], s['player'], s['batting_team'], s['runs'], s['balls_faced'], s['fours'], s['sixes'], s['dots_faced'], s['dismissed'], s['strike_rate'], s['bowling_team'], s['overs'], s['balls_bowled'], s['runs_conceded'], s['wickets'], s['wides'], s['noballs'], s['dots_bowled'], s['economy']) for s in stats])
    
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table_name}_player ON {table_name}(player)")
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table_name}_venue ON {table_name}(venue)")
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table_name}_batting_team ON {table_name}(batting_team)")
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table_name}_bowling_team ON {table_name}(bowling_team)")
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table_name}_date ON {table_name}(date)")
    
    conn.commit()
    conn.close()
    
    if verbose:
        print(f"Saved {len(stats)} player records to {table_name} table in {db_path}")


def process_and_save_player_stats(project_root: Path, gender: str = 'male', table_name: str = 'first_innings_player_stats', verbose: bool = True) -> None:
    """Complete workflow: load matches, extract combined player stats (batting and bowling), and save to database."""
    from .match_loader import load_filtered_matches
    
    if verbose:
        print("=" * 60)
        print("EXTRACTING PLAYER STATISTICS FROM FIRST INNINGS")
        print("=" * 60)
    
    matches = load_filtered_matches(project_root, gender=gender, verbose=verbose)
    
    if verbose:
        print(f"\\nExtracting combined player stats from {len(matches)} matches...")
    
    player_stats = extract_first_innings_combined_stats(matches)
    
    if verbose:
        print(f"Extracted stats for {len(player_stats)} player records")
        unique_players = len(set(s['player'] for s in player_stats))
        players_who_batted = sum(1 for s in player_stats if s['runs'] is not None)
        players_who_bowled = sum(1 for s in player_stats if s['wickets'] is not None)
        players_both = sum(1 for s in player_stats if s['runs'] is not None and s['wickets'] is not None)
        print(f"Unique players: {unique_players}")
        print(f"Player records with batting: {players_who_batted}")
        print(f"Player records with bowling: {players_who_bowled}")
        print(f"Player records with both: {players_both}")
    
    db_path = project_root / 't20s_json' / 'venue_metrics.db'
    save_player_stats_to_db(player_stats, db_path, table_name=table_name, verbose=verbose)
    
    if verbose:
        print("=" * 60)
        print("COMPLETED")
        print("=" * 60)
'''

# Write the file
project_root = Path().resolve().parent if Path().resolve().name == 'research' else Path().resolve()
file_path = project_root / 'forecaster' / 'player_stats.py'
with open(file_path, 'w') as f:
    f.write(player_stats_code)

print(f"Created {file_path}")

Created /Users/damith/projects/t20_score_prediction/forecaster/player_stats.py


In [ ]:
# Recreate with schema that supports both batting and bowling
import sqlite3
from pathlib import Path

project_root = Path().resolve().parent if Path().resolve().name == 'research' else Path().resolve()
db_path = project_root / 't20s_json' / 'venue_metrics.db'

conn = sqlite3.connect(str(db_path))
cursor = conn.cursor()

# Rename old table as backup
try:
    cursor.execute("ALTER TABLE first_innings_player_stats RENAME TO first_innings_player_stats_old")
    print("Renamed old table to first_innings_player_stats_old")
except Exception as e:
    print(f"Could not rename: {e}")

conn.commit()
conn.close()

print("Ready to create new table")

Could not rename: database is locked
Ready to create new table


: 

# Player Statistics Extraction

This notebook extracts player-level statistics (batting and bowling) from first innings of T20 matches and stores them in the `venue_metrics.db` database.

## What it does:
1. Uses `forecaster.match_loader` to load filtered match data
2. Extracts individual player **batting** statistics from first innings:
   - Runs scored
   - Balls faced
   - Boundaries (4s and 6s)
   - Dot balls faced
   - Strike rate
   - Whether dismissed
3. Extracts individual player **bowling** statistics from first innings:
   - Overs bowled
   - Runs conceded
   - Wickets taken
   - Extras (wides, no-balls)
   - Dot balls bowled
   - Economy rate
4. Saves data to two tables in SQLite database

## Database Tables

### `first_innings_player_stats` (Batting)
- **match_id**: Match identifier
- **date**: Match date
- **venue**: Match venue
- **team**: Team name
- **player**: Player name
- **runs**: Runs scored
- **balls**: Balls faced
- **fours**: Number of fours
- **sixes**: Number of sixes
- **dots**: Dot balls faced
- **dismissed**: Whether dismissed (1/0)
- **strike_rate**: Strike rate

### `first_innings_bowling_stats` (Bowling)
- **match_id**: Match identifier
- **date**: Match date
- **venue**: Match venue
- **team**: Bowling team name
- **bowler**: Bowler name
- **overs**: Overs bowled (e.g., 3.4)
- **balls**: Total balls bowled
- **runs_conceded**: Runs conceded
- **wickets**: Wickets taken (excluding run-outs)
- **wides**: Wide deliveries
- **noballs**: No-ball deliveries
- **dots**: Dot balls bowled
- **economy**: Economy rate (runs per over)

In [5]:
import sys
from pathlib import Path

# Add project root to path
current_dir = Path().resolve()
if current_dir.name == 'research':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Force reload of forecaster modules
import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith('forecaster'):
        del sys.modules[mod]

# Import forecaster module
from forecaster import process_and_save_player_stats

# Process and save player stats (batting and bowling) to database
process_and_save_player_stats(project_root, gender='male', verbose=True)

EXTRACTING PLAYER STATISTICS FROM ALL INNINGS
LOADING AND FILTERING T20 MATCH DATA
Found 3112 JSON files in male directory
Successfully loaded 3112 matches
Filtered 'no result' matches: 3112 → 3038 (excluded: 74)
Filtered incomplete innings: 3038 → 2923 (excluded: 115)
Final dataset: 2923 matches

Extracting combined player stats from 2923 matches...
Extracted stats for 61306 player records
Unique players: 4223
Player records with batting: 47673
Player records with bowling: 35195
Player records with both batting and bowling: 21562
Saved 61306 player records to player_stats table in /Users/damith/projects/t20_score_prediction/t20s_json/venue_metrics.db
COMPLETED


In [2]:
# Verify the data was saved correctly
import sqlite3
import pandas as pd

db_path = project_root / 't20s_json' / 'venue_metrics.db'
conn = sqlite3.connect(str(db_path))

# Check table structure
print("Table Schema:")
print("=" * 60)
schema = pd.read_sql_query("PRAGMA table_info(first_innings_player_stats)", conn)
print(schema.to_string(index=False))

# Check row count
print("\n" + "=" * 60)
row_count = pd.read_sql_query("SELECT COUNT(*) as count FROM first_innings_player_stats", conn)
print(f"Total rows: {row_count['count'][0]}")

# Check unique players
unique_players = pd.read_sql_query("SELECT COUNT(DISTINCT player) as count FROM first_innings_player_stats", conn)
print(f"Unique players: {unique_players['count'][0]}")

# Show sample data
print("\n" + "=" * 60)
print("Sample data (top 10 by runs):")
print("=" * 60)
sample = pd.read_sql_query("""
    SELECT player, team, venue, runs, balls, fours, sixes, strike_rate
    FROM first_innings_player_stats
    ORDER BY runs DESC
    LIMIT 10
""", conn)
print(sample.to_string(index=False))

conn.close()

Table Schema:
 cid        name    type  notnull dflt_value  pk
   0    match_id    TEXT        0       None   0
   1        date    TEXT        0       None   0
   2       venue    TEXT        0       None   0
   3        team    TEXT        0       None   0
   4      player    TEXT        0       None   0
   5        runs INTEGER        0       None   0
   6       balls INTEGER        0       None   0
   7       fours INTEGER        0       None   0
   8       sixes INTEGER        0       None   0
   9        dots INTEGER        0       None   0
  10   dismissed INTEGER        0       None   0
  11 strike_rate    REAL        0       None   0

Total rows: 25126
Unique players: 3547

Sample data (top 10 by runs):
         player           team                                                       venue  runs  balls  fours  sixes  strike_rate
       AJ Finch      Australia                                          Harare Sports Club   172     76     16     10       226.32
 Mohammad Ihsan 

In [3]:
# Query player statistics - Example: Top scorers at specific venues
import sqlite3
import pandas as pd

conn = sqlite3.connect(str(db_path))

print("Top 20 Run Scorers in First Innings (All Venues):")
print("=" * 80)
top_scorers = pd.read_sql_query("""
    SELECT 
        player,
        COUNT(*) as matches,
        SUM(runs) as total_runs,
        SUM(balls) as total_balls,
        SUM(fours) as total_fours,
        SUM(sixes) as total_sixes,
        ROUND(AVG(runs), 2) as avg_runs,
        ROUND((SUM(runs) * 100.0 / SUM(balls)), 2) as career_strike_rate
    FROM first_innings_player_stats
    GROUP BY player
    ORDER BY total_runs DESC
    LIMIT 20
""", conn)
print(top_scorers.to_string(index=False))

conn.close()

Top 20 Run Scorers in First Innings (All Venues):
         player  matches  total_runs  total_balls  total_fours  total_sixes  avg_runs  career_strike_rate
      RG Sharma       78        2499         1710          226          120     32.04              146.14
     Babar Azam       62        2342         1812          237           40     37.77              129.25
        V Kohli       64        1966         1450          178           61     30.72              135.59
     MJ Guptill       59        1953         1399          179           89     33.10              139.60
     JC Buttler       54        1846         1223          159           87     34.19              150.94
Mohammad Rizwan       42        1762         1345          150           49     41.95              131.00
       SA Yadav       55        1659          982          140           99     30.16              168.94
      Q de Kock       57        1604         1157          162           62     28.14              138

In [4]:
# Example: Player performance at specific venues
import sqlite3
import pandas as pd

conn = sqlite3.connect(str(db_path))

print("Top Scorers at Wankhede Stadium, Mumbai:")
print("=" * 80)
venue_scorers = pd.read_sql_query("""
    SELECT 
        player,
        COUNT(*) as innings,
        SUM(runs) as total_runs,
        ROUND(AVG(runs), 2) as avg_runs,
        MAX(runs) as highest_score,
        ROUND(AVG(strike_rate), 2) as avg_strike_rate
    FROM first_innings_player_stats
    WHERE venue LIKE '%Wankhede%'
    GROUP BY player
    HAVING innings >= 2
    ORDER BY total_runs DESC
    LIMIT 15
""", conn)
print(venue_scorers.to_string(index=False))

print("\n\n" + "=" * 80)
print("Players with highest strike rates (min 500 balls faced):")
print("=" * 80)
high_sr = pd.read_sql_query("""
    SELECT 
        player,
        COUNT(*) as innings,
        SUM(runs) as total_runs,
        SUM(balls) as total_balls,
        ROUND((SUM(runs) * 100.0 / SUM(balls)), 2) as career_strike_rate,
        SUM(sixes) as total_sixes
    FROM first_innings_player_stats
    GROUP BY player
    HAVING SUM(balls) >= 500
    ORDER BY career_strike_rate DESC
    LIMIT 15
""", conn)
print(high_sr.to_string(index=False))

conn.close()

Top Scorers at Wankhede Stadium, Mumbai:
   player  innings  total_runs  avg_runs  highest_score  avg_strike_rate
  V Kohli        3         197     65.67             89           206.91
RG Sharma        3         138     46.00             71           157.95
 MS Dhoni        2          53     26.50             38           188.89
 AR Patel        2          46     23.00             31           145.68
AM Rahane        2          43     21.50             40            87.15
HH Pandya        2          38     19.00             29           128.70
SV Samson        2          21     10.50             16           155.95
 SA Yadav        2           9      4.50              7            68.34


Players with highest strike rates (min 500 balls faced):
         player  innings  total_runs  total_balls  career_strike_rate  total_sixes
       TH David       25         917          538              170.45           52
       SA Yadav       55        1659          982              168.94        

In [7]:
# Verify bowling stats table
import sqlite3
import pandas as pd

conn = sqlite3.connect(str(db_path))

# Check table structure
print("Bowling Stats Table Schema:")
print("=" * 60)
schema = pd.read_sql_query("PRAGMA table_info(first_innings_bowling_stats)", conn)
print(schema.to_string(index=False))

# Check row count
print("\n" + "=" * 60)
row_count = pd.read_sql_query("SELECT COUNT(*) as count FROM first_innings_bowling_stats", conn)
print(f"Total bowling records: {row_count['count'][0]}")

# Check unique bowlers
unique_bowlers = pd.read_sql_query("SELECT COUNT(DISTINCT bowler) as count FROM first_innings_bowling_stats", conn)
print(f"Unique bowlers: {unique_bowlers['count'][0]}")

# Show sample data
print("\n" + "=" * 60)
print("Sample bowling data (best figures by wickets):")
print("=" * 60)
sample = pd.read_sql_query("""
    SELECT bowler, team, venue, overs, runs_conceded, wickets, economy
    FROM first_innings_bowling_stats
    ORDER BY wickets DESC, runs_conceded ASC
    LIMIT 10
""", conn)
print(sample.to_string(index=False))

conn.close()

Bowling Stats Table Schema:
 cid          name    type  notnull dflt_value  pk
   0      match_id    TEXT        0       None   0
   1          date    TEXT        0       None   0
   2         venue    TEXT        0       None   0
   3          team    TEXT        0       None   0
   4        bowler    TEXT        0       None   0
   5         overs    REAL        0       None   0
   6         balls INTEGER        0       None   0
   7 runs_conceded INTEGER        0       None   0
   8       wickets INTEGER        0       None   0
   9         wides INTEGER        0       None   0
  10       noballs INTEGER        0       None   0
  11          dots INTEGER        0       None   0
  12       economy    REAL        0       None   0

Total bowling records: 17811
Unique bowlers: 2546

Sample bowling data (best figures by wickets):
           bowler        team                                  venue  overs  runs_conceded  wickets  economy
    Syazrul Idrus    Malaysia            Bayuemas 

In [8]:
# Bowling statistics - Top wicket takers and economy rates
import sqlite3
import pandas as pd

conn = sqlite3.connect(str(db_path))

print("Top 20 Wicket Takers in First Innings:")
print("=" * 100)
top_wicket_takers = pd.read_sql_query("""
    SELECT 
        bowler,
        COUNT(*) as matches,
        ROUND(SUM(overs), 1) as total_overs,
        SUM(wickets) as total_wickets,
        SUM(runs_conceded) as total_runs,
        ROUND(AVG(wickets), 2) as avg_wickets,
        ROUND(AVG(economy), 2) as avg_economy,
        ROUND(SUM(runs_conceded) * 1.0 / SUM(wickets), 2) as bowling_average
    FROM first_innings_bowling_stats
    GROUP BY bowler
    ORDER BY total_wickets DESC
    LIMIT 20
""", conn)
print(top_wicket_takers.to_string(index=False))

print("\n\n" + "=" * 100)
print("Best Economy Rates (min 30 overs bowled):")
print("=" * 100)
best_economy = pd.read_sql_query("""
    SELECT 
        bowler,
        COUNT(*) as matches,
        ROUND(SUM(overs), 1) as total_overs,
        SUM(wickets) as total_wickets,
        SUM(runs_conceded) as total_runs,
        ROUND(AVG(economy), 2) as avg_economy,
        ROUND(SUM(runs_conceded) * 1.0 / SUM(wickets), 2) as bowling_average
    FROM first_innings_bowling_stats
    GROUP BY bowler
    HAVING SUM(overs) >= 30
    ORDER BY avg_economy ASC
    LIMIT 20
""", conn)
print(best_economy.to_string(index=False))

conn.close()

Top 20 Wicket Takers in First Innings:
             bowler  matches  total_overs  total_wickets  total_runs  avg_wickets  avg_economy  bowling_average
  Mustafizur Rahman       58        225.1             85        1738         1.47         7.66            20.45
          AU Rashid       66        249.2             77        1850         1.17         7.67            24.03
         TG Southee       56        219.0             74        1782         1.32         8.18            24.08
            A Zampa       60        230.0             70        1740         1.17         7.53            24.86
          CJ Jordan       53        196.5             69        1742         1.30         8.78            25.25
         Haris Rauf       41        156.3             68        1222         1.66         7.74            17.97
       S Lamichhane       38        145.5             67         871         1.76         6.17            13.00
       PWH de Silva       37        144.0             66        1

In [8]:
# Classify players based on their statistics
import sys
from pathlib import Path

# Add project root to path
current_dir = Path().resolve()
if current_dir.name == 'research':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Force reload of forecaster modules
import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith('forecaster'):
        del sys.modules[mod]

# Import forecaster module
from forecaster import process_player_classifications

# Classify players
process_player_classifications(project_root, source_table='player_stats', target_table='player_classifications', verbose=True)

AGGREGATING AND CLASSIFYING PLAYERS

Aggregating player statistics from player_stats...
Aggregated stats for 4223 players
Classified 4223 players

Classification Summary:
  INSUFFICIENT_DATA: 3013
  UTILITY_PLAYER: 836
  BOWLER: 231
  BATTER: 87
  ALL_ROUNDER: 56

Saved 4223 player classifications to player_classifications table
COMPLETED


In [7]:
# View classification summary by category
import sqlite3
from pathlib import Path

project_root = Path().resolve().parent if Path().resolve().name == 'research' else Path().resolve()
db_path = project_root / 't20s_json' / 'venue_metrics.db'

conn = sqlite3.connect(str(db_path))

print("=" * 80)
print("PLAYER CLASSIFICATION SUMMARY")
print("=" * 80)

# Summary by classification
query = """
SELECT 
    classification,
    COUNT(*) as count,
    AVG(matches) as avg_matches,
    AVG(bat_avg) as avg_bat_avg,
    AVG(bat_sr) as avg_bat_sr,
    AVG(bowl_avg) as avg_bowl_avg,
    AVG(econ) as avg_econ
FROM player_classifications
GROUP BY classification
ORDER BY count DESC
"""

import pandas as pd
df_summary = pd.read_sql_query(query, conn)
print("\nSummary Statistics by Classification:")
print(df_summary.to_string(index=False))

# Top 5 players from each main category
print("\n" + "=" * 80)
print("TOP PLAYERS BY CATEGORY")
print("=" * 80)

for category in ['BATTER', 'BOWLER', 'ALL_ROUNDER']:
    print(f"\nTop 5 {category}s:")
    if category == 'BATTER':
        order_by = 'total_runs DESC'
    elif category == 'BOWLER':
        order_by = 'total_wickets DESC'
    else:
        order_by = 'matches DESC'
    
    query = f"""
    SELECT player, matches, total_runs, bat_avg, bat_sr, total_wickets, bowl_avg, econ
    FROM player_classifications
    WHERE classification = '{category}'
    ORDER BY {order_by}
    LIMIT 5
    """
    df_top = pd.read_sql_query(query, conn)
    print(df_top.to_string(index=False))

conn.close()

PLAYER CLASSIFICATION SUMMARY

Summary Statistics by Classification:
   classification  count  avg_matches  avg_bat_avg  avg_bat_sr  avg_bowl_avg  avg_econ
INSUFFICIENT_DATA   3013     5.638566    10.415904   76.777806    488.966857  5.729426
   UTILITY_PLAYER    886    34.981941    19.713352  112.470609    270.878239  6.326174
           BOWLER    209    37.129187     9.212057   87.436507     19.782057  6.851866
           BATTER     79    50.354430    34.765949  140.145316    790.974684  3.369241
      ALL_ROUNDER     36    44.027778    29.549167  141.558333     20.926944  6.930000

TOP PLAYERS BY CATEGORY

Top 5 BATTERs:
         player  matches  total_runs  bat_avg  bat_sr  total_wickets  bowl_avg  econ
     Babar Azam      125        4316    40.34  128.45              0    999.00  0.00
      RG Sharma      137        3886    31.59  141.10              1    106.00 10.39
        V Kohli      107        3823    49.01  135.66              4     51.75  8.35
     JC Buttler      123    